In [1]:
!pip install python-telegram-bot[webhooks] pyngrok nest_asyncio sentence-transformers groq sqlite-vec -q

In [12]:
!pip install openai -q

ERROR:asyncio:Task was destroyed but it is pending!
task: <Task pending name='Application:8658999161:update_fetcher' coro=<Application._update_fetcher() running at /usr/local/lib/python3.12/dist-packages/telegram/ext/_application.py:1233> wait_for=<Future pending cb=[Task.__wakeup()]>>


In [3]:
import os
os.makedirs("docs", exist_ok=True)

docs = {
    "docs/faq.md": """# General FAQ

## What is this service?
This is an AI-powered assistant that answers questions from our knowledge base.

## How do I contact support?
You can reach support at support@company.com or call +1-800-555-0199 between 9am and 5pm EST.

## What are your business hours?
We are open Monday to Friday, 9am to 6pm EST. We are closed on weekends and public holidays.

## How long does shipping take?
Standard shipping takes 5-7 business days. Express shipping takes 1-2 business days.
""",

    "docs/policies.md": """# Company Policies

## Refund Policy
We offer a 30-day money-back guarantee on all products. To request a refund, email refunds@company.com with your order number.

## Privacy Policy
We do not sell or share your personal data with third parties. All data is encrypted and stored securely.

## Account Policy
Users must be 18 or older to create an account. Accounts inactive for 12 months may be deactivated.

## Payment Policy
We accept Visa, Mastercard, PayPal, and bank transfers. All payments are processed securely via Stripe.
""",

    "docs/products.md": """# Product Information

## Product A — Basic Plan
Price: $9.99/month. Includes 10GB storage, email support, and access to core features.

## Product B — Pro Plan
Price: $29.99/month. Includes 100GB storage, priority support, API access, and advanced analytics.

## Product C — Enterprise Plan
Price: Custom pricing. Includes unlimited storage, dedicated support, SLA guarantee, and custom integrations.

## Free Trial
All plans come with a 14-day free trial. No credit card required to start.
""",

    "docs/technical.md": """# Technical Documentation

## System Requirements
Our platform requires Python 3.8 or higher. Supported operating systems: Windows 10+, macOS 11+, Ubuntu 20.04+.

## API Rate Limits
Free tier: 100 requests per hour. Pro tier: 1000 requests per hour. Enterprise: unlimited.

## Authentication
We use OAuth 2.0 for authentication. API keys can be generated from your dashboard under Settings > API.

## Webhooks
Webhooks are supported on Pro and Enterprise plans. Events include: payment.success, user.created, and order.shipped.
"""
}

for path, content in docs.items():
    with open(path, "w") as f:
        f.write(content)

print("✅ Created 4 documents in /docs folder:")
for path in docs:
    print(f"   {path}")

✅ Created 4 documents in /docs folder:
   docs/faq.md
   docs/policies.md
   docs/products.md
   docs/technical.md


In [4]:
import sqlite3
import sqlite_vec
import json
import os
import hashlib
from sentence_transformers import SentenceTransformer

EMBED_MODEL = SentenceTransformer("all-MiniLM-L6-v2")
DB_PATH = "rag.db"
CHUNK_SIZE = 200  # words per chunk
CHUNK_OVERLAP = 30

def chunk_text(text, source, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    words = text.split()
    chunks = []
    i = 0
    while i < len(words):
        chunk = " ".join(words[i:i+chunk_size])
        chunks.append({"text": chunk, "source": source})
        i += chunk_size - overlap
    return chunks

def get_embedding(text):
    return EMBED_MODEL.encode(text).tolist()

def init_db():
    conn = sqlite3.connect(DB_PATH)
    conn.enable_load_extension(True)
    sqlite_vec.load(conn)
    conn.enable_load_extension(False)
    conn.execute("""
        CREATE TABLE IF NOT EXISTS chunks (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            source TEXT,
            text TEXT,
            embedding BLOB
        )
    """)
    conn.commit()
    return conn

def ingest_docs(docs_folder="docs"):
    conn = init_db()
    total = 0
    for filename in os.listdir(docs_folder):
        if not filename.endswith((".md", ".txt")):
            continue
        filepath = os.path.join(docs_folder, filename)
        with open(filepath, "r") as f:
            text = f.read()
        chunks = chunk_text(text, source=filename)
        for chunk in chunks:
            embedding = get_embedding(chunk["text"])
            conn.execute(
                "INSERT INTO chunks (source, text, embedding) VALUES (?, ?, ?)",
                (chunk["source"], chunk["text"], json.dumps(embedding))
            )
            total += 1
    conn.commit()
    conn.close()
    print(f"✅ Ingested {total} chunks from {docs_folder}/")

ingest_docs()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Ingested 4 chunks from docs/


In [25]:
from openai import OpenAI
import numpy as np
import sqlite3
import json

OPENROUTER_API_KEY = "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"

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY
)

query_cache = {}

def cosine_similarity(a, b):
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

def retrieve(query, top_k=3):
    query_vec = get_embedding(query)
    conn = sqlite3.connect(DB_PATH)
    rows = conn.execute("SELECT source, text, embedding FROM chunks").fetchall()
    conn.close()
    scored = []
    for source, text, emb_json in rows:
        emb = json.loads(emb_json)
        score = cosine_similarity(query_vec, emb)
        scored.append((score, source, text))
    scored.sort(reverse=True)
    return scored[:top_k]

def ask_mistral(context, query):
    response = client.chat.completions.create(
        model="openrouter/auto",
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a helpful assistant. Use only the context below to answer.\n"
                    "If the answer is not in the context, say 'I don't have information on that.'\n\n"
                    f"Context:\n{context}"
                )
            },
            {"role": "user", "content": query}
        ],
        max_tokens=512,
        temperature=0.3
    )
    return response.choices[0].message.content.strip()

def rag_answer(query):
    cache_key = query.strip().lower()
    if cache_key in query_cache:
        print("Cache hit!")
        return query_cache[cache_key]

    chunks = retrieve(query)
    context = "\n\n---\n\n".join([f"[{src}]\n{txt}" for _, src, txt in chunks])
    answer = ask_mistral(context, query)

    sources = list(dict.fromkeys([src for _, src, _ in chunks]))
    full_answer = answer + f"\n\n📄 Sources: {', '.join(sources)}"

    query_cache[cache_key] = full_answer
    return full_answer

print("✅ RAG pipeline ready with Mistral-7B via OpenRouter!")

✅ RAG pipeline ready with Mistral-7B via OpenRouter!


In [ ]:
import nest_asyncio
nest_asyncio.apply()

from telegram import Update
from telegram.ext import ApplicationBuilder, CommandHandler, ContextTypes
from collections import defaultdict, deque
import asyncio

BOT_TOKEN = "8658999161:AAGZiu9zpqglgvONR93m9sfexAGx6uWr0PE"

user_history = defaultdict(lambda: deque(maxlen=3))

async def start(update: Update, context: ContextTypes.DEFAULT_TYPE):
    await update.message.reply_text(
        "👋 Hello! I'm your RAG assistant.\n\n"
        "Use /ask <question> to search the knowledge base.\n"
        "Use /help for more info."
    )

async def help_command(update: Update, context: ContextTypes.DEFAULT_TYPE):
    await update.message.reply_text(
        "📖 Commands:\n\n"
        "/ask <query> — search the knowledge base\n"
        "/history — show your last 3 questions\n"
        "/image — send an image (coming soon)\n"
        "/help — show this message"
    )

async def ask(update: Update, context: ContextTypes.DEFAULT_TYPE):
    if not context.args:
        await update.message.reply_text(
            "Please provide a question.\n"
            "Example: /ask What is the refund policy?"
        )
        return
    query = " ".join(context.args)
    user_id = update.effective_user.id
    user_history[user_id].append(query)
    await update.message.reply_text("🔍 Searching knowledge base...")
    try:
        answer = rag_answer(query)
        await update.message.reply_text(answer)
    except Exception as e:
        await update.message.reply_text(f"❌ Error: {str(e)}")

async def history_command(update: Update, context: ContextTypes.DEFAULT_TYPE):
    user_id = update.effective_user.id
    hist = list(user_history[user_id])
    if not hist:
        await update.message.reply_text("No history yet. Use /ask to get started!")
        return
    lines = "\n".join([f"{i+1}. {q}" for i, q in enumerate(hist)])
    await update.message.reply_text(f"🕓 Your last questions:\n\n{lines}")

async def image_command(update: Update, context: ContextTypes.DEFAULT_TYPE):
    await update.message.reply_text("🖼️ Image support coming soon! Use /ask <query> for now.")

async def main():
    # First delete any existing webhook so polling works
    app = ApplicationBuilder().token(BOT_TOKEN).build()
    await app.bot.delete_webhook(drop_pending_updates=True)

    app.add_handler(CommandHandler("start",   start))
    app.add_handler(CommandHandler("help",    help_command))
    app.add_handler(CommandHandler("ask",     ask))
    app.add_handler(CommandHandler("history", history_command))
    app.add_handler(CommandHandler("image",   image_command))

    print("🤖 Bot is running via polling! Go test it on Telegram.")

    # Polling — no ngrok needed, works perfectly in Colab
    await app.run_polling(drop_pending_updates=True)

asyncio.get_event_loop().run_until_complete(main())

🤖 Bot is running via polling! Go test it on Telegram.


In [18]:
test = ask_mistral("This is a test context.", "Say hello in one sentence.")
print(test)

RateLimitError: Error code: 429 - {'error': {'message': 'Provider returned error', 'code': 429, 'metadata': {'raw': 'mistralai/mistral-small-3.1-24b-instruct:free is temporarily rate-limited upstream. Please retry shortly, or add your own key to accumulate your rate limits: https://openrouter.ai/settings/integrations', 'provider_name': 'Venice', 'is_byok': False}}, 'user_id': 'user_3BDLdJUh5b2bKhgER8kscQkI7fV'}